this notebook takes the output of feature_engineering.ipynb as input. Target encoding was unable to occur in feature engineering to prevent leakage into $Y$ and must happen at the cross fitting stage. 

In [ ]:
seed = 42


# Instantiate encoder
# target encoding needs to happen at each kfold
encoder = TargetEncoder(categories='auto',
                        target_type='binary',
                        smooth='auto',
                        cv=5
                        )


# generate 5 reproducible splits
kf = KFold(n_splits=5,
                    shuffle=True,
                    random_state=seed)


Y_res = pd.Series(np.zeros(len(sample_df)), index=sample_df.index)
D_res = pd.Series(np.zeros(len(sample_df)), index=sample_df.index)

d = sample_df['payprice'].astype(float)
y = sample_df['click'].astype(int)

fold = 0
for train_index, test_index in kf.split(sample_df):

  print(fold)
  fold += 1
  # complete target encoding
  X_target_encoders = sample_df[target_categorical_features].astype(str)
  #d = sample_df['creative'].astype(int)
  #d = pd.Series(le.fit_transform(sample_df['creative']), index=sample_df.index) # encodes 8 creatives as 0-7


  X_train_target_encoders = X_target_encoders.iloc[train_index]
  X_test_target_encoders = X_target_encoders.iloc[test_index]

  d_train = d.iloc[train_index]
  d_test = d.iloc[test_index]

  y_train = y.iloc[train_index]
  y_test = y.iloc[test_index]

  # fit train data
  encoder.fit(X_train_target_encoders, y_train)

  # transform both train and test
  X_train_encoded = encoder.transform(X_train_target_encoders)
  X_test_encoded = encoder.transform(X_test_target_encoders)

  # join the target encoded data back with the rest of the data
  other_columns_train = sample_df.drop(columns=target_categorical_features + ['click'] + ['creative']).iloc[train_index]
  other_columns_test  = sample_df.drop(columns=target_categorical_features + ['click'] + ['creative']).iloc[test_index]
  X_train = X_train_encoded.join(other_columns_train, how = 'inner')
  X_test  = X_test_encoded.join(other_columns_test,  how = 'inner')

  regressor = RandomForestRegressor(
      n_estimators=100,
      random_state=seed,
      n_jobs=-1,
      oob_score=False
  )

  classifier = RandomForestClassifier(
      n_estimators=100,
      random_state=seed,
      class_weight='balanced',
      n_jobs=-1,
      oob_score=False
  )


  # fit a RF Regressor on the training X and training Y
  regressor.fit(X_train, y_train)
  y_pred = regressor.predict(X_test)

  Y_res.iloc[test_index] = y_test - y_pred

  classifier.fit(X_train, d_train)
  # predict class probabilities for D
  # we take the second col, where P(D=1|X)
  #D_prob = classifier.predict_proba(X_test)[:,1]
  d_pred = regressor.predict(X_test)
  #D_prob = classifier.predict_proba(X_test)

  # one-hot encode d_test then subtract predicted probabilities
  #D_onehot = pd.get_dummies(d_test).reindex(columns=range(n_classes), fill_value=0)
  #D_res.iloc[test_index] = D_onehot.values - D_prob
  # computed as the mean predicted class probabilities of the trees
  # in the forest. The class prob of a single tree is the fraction
  # of samples of the same class in a leaf
  D_res.iloc[test_index] = d_test - d_pred
  # adding to propensity score for heteroskedastic plot
  #propensity_score.iloc[test_index] = D_prob
  #propensity_scores.iloc[test_index] = D_prob

  #D_res.iloc[test_index] = d_test - D_prob

#model = sm.OLS(Y_res, D_res).fit(cov_type='HC3')

In [ ]:
# final OLS on residuals
model = sm.OLS(Y_res.values, sm.add_constant(D_res.values)).fit(cov_type='HC3')
print(model.summary())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(D_res, bins=50, edgecolor='k')
axes[0].axvline(0, color='red', linewidth=0.8)
axes[0].set_title('D Residuals Distribution')
axes[0].set_xlabel('payprice residuals')

axes[1].scatter(D_res, Y_res, alpha=0.05, s=1)
axes[1].axhline(0, color='red', linewidth=0.8)
axes[1].axvline(0, color='red', linewidth=0.8)
axes[1].set_title('Residuals Plot')
axes[1].set_xlabel('D residuals')
axes[1].set_ylabel('Y residuals')

axes[2].hist(sample_df['payprice'], bins=50, edgecolor='k')
axes[2].set_title('Raw payprice Distribution')
axes[2].set_xlabel('payprice')

plt.tight_layout()
plt.show()